In [1]:
import numpy as np

g = 9.81  # gravity (m/s^2)

def wave_number(T, h):
    """Solve dispersion relation for wave number k using Newton-Raphson"""
    omega = 2*np.pi/T
    k = omega**2 / g  # initial guess
    
    for _ in range(100):
        f = g*k*np.tanh(k*h) - omega**2
        df = g*np.tanh(k*h) + g*k*h*(1/np.cosh(k*h))**2
        k = k - f/df
    return k

def wave_kinematics(x, z, t, H, T, h):
    """Airy wave theory velocities and accelerations"""
    omega = 2*np.pi/T
    k = wave_number(T, h)
    
    a = H/2
    cosh_term = np.cosh(k*(z+h)) / np.sinh(k*h)
    sinh_term = np.sinh(k*(z+h)) / np.sinh(k*h)
    
    phase = k*x - omega*t
    
    u = a * omega * cosh_term * np.cos(phase)          # horizontal velocity
    w = a * omega * sinh_term * np.sin(phase)          # vertical velocity
    
    u_dot = a * omega**2 * cosh_term * np.sin(phase)   # horizontal acceleration
    w_dot = -a * omega**2 * sinh_term * np.cos(phase)  # vertical acceleration
    
    return u, w, u_dot, w_dot

def morison_force_member(
        start, end,
        H, T, h,
        D,
        rho=1025,
        CD=1.0,
        CM=2.0,
        t=0.0,
        n_segments=50
    ):
    """
    Computes total hydrodynamic force on a cylindrical member
    using Morison equation.
    
    start, end : (x,z) coordinates
    """
    
    start = np.array(start)
    end = np.array(end)
    
    L_vec = end - start
    L = np.linalg.norm(L_vec)
    e_m = L_vec / L  # unit vector along member
    
    Fx_total = 0
    Fz_total = 0
    
    # Discretize member
    for i in range(n_segments):
        s = (i + 0.5)/n_segments
        point = start + s*L_vec
        x, z = point
        
        # Only if submerged
        if z <= 0:
            u, w, u_dot, w_dot = wave_kinematics(x, z, t, H, T, h)
            
            vel = np.array([u, w])
            acc = np.array([u_dot, w_dot])
            
            # Normal components
            vel_n = vel - np.dot(vel, e_m)*e_m
            acc_n = acc - np.dot(acc, e_m)*e_m
            
            vel_n_mag = np.linalg.norm(vel_n)
            
            # Morison per unit length (vector form)
            f_drag = 0.5 * rho * CD * D * vel_n_mag * vel_n
            f_inertia = rho * CM * (np.pi*D**2/4) * acc_n
            
            f_total = f_drag + f_inertia
            
            dL = L / n_segments
            Fx_total += f_total[0] * dL
            Fz_total += f_total[1] * dL
    
    return Fx_total, Fz_total

# -------------------------------
# Example usage
# -------------------------------

if __name__ == "__main__":
    
    start = (0, -30)      # x,z (m)
    end   = (10, -5)      # diagonal member
    
    H = 6.0               # wave height (m)
    T = 10.0              # wave period (s)
    h = 50.0              # water depth (m)
    
    D = 1.5               # member diameter (m)
    
    Fx, Fz = morison_force_member(
        start, end,
        H, T, h,
        D,
        rho=1025,
        CD=1.0,
        CM=2.0,
        t=5.0
    )
    
    print("Total Hydrodynamic Forces:")
    print(f"Fx = {Fx:.2f} N")
    print(f"Fz = {Fz:.2f} N")


Total Hydrodynamic Forces:
Fx = -45305.11 N
Fz = 18122.05 N


In [3]:
import numpy as np

g = 9.81  # gravity (m/s^2)

# ------------------------
# Wave & Kinematics
# ------------------------
def wave_number(T, h):
    """Solve dispersion relation for wave number k using Newton-Raphson"""
    omega = 2*np.pi/T
    k = omega**2 / g  # initial guess
    for _ in range(100):
        f = g*k*np.tanh(k*h) - omega**2
        df = g*np.tanh(k*h) + g*k*h*(1/np.cosh(k*h))**2
        k = k - f/df
    return k

def wave_kinematics(x, z, t, H, T, h, zeta=0):
    """Airy wave theory velocities and accelerations with Wheeler stretching"""
    omega = 2*np.pi/T
    k = wave_number(T, h)
    
    # Wheeler stretching: stretch velocity profile up to wave crest
    z_stretch = (z + h) * (h + zeta) / h - h
    
    a = H/2
    cosh_term = np.cosh(k*(z_stretch+h)) / np.sinh(k*h)
    sinh_term = np.sinh(k*(z_stretch+h)) / np.sinh(k*h)
    
    phase = k*x - omega*t
    
    u = a * omega * cosh_term * np.cos(phase)          # horizontal velocity
    w = a * omega * sinh_term * np.sin(phase)          # vertical velocity
    u_dot = a * omega**2 * cosh_term * np.sin(phase)   # horizontal acceleration
    w_dot = -a * omega**2 * sinh_term * np.cos(phase)  # vertical acceleration
    
    return u, w, u_dot, w_dot

# ------------------------
# Morison force on a member segment
# ------------------------
def morison_segment_force(vel_n, acc_n, D, rho=1025, CD=1.0, CM=2.0):
    vel_mag = np.linalg.norm(vel_n)
    f_drag = 0.5 * rho * CD * D * vel_mag * vel_n
    f_inertia = rho * CM * (np.pi * D**2 / 4) * acc_n
    return f_drag + f_inertia

# ------------------------
# Compute member properties
# ------------------------
def member_properties(nodes, member):
    start_node = np.array(nodes[member['start']])
    end_node = np.array(nodes[member['end']])
    vec = end_node - start_node
    L = np.linalg.norm(vec)
    e_m = vec / L  # unit vector along member
    
    # Orientation angles for diagonal members
    dx, dy, dz = vec
    theta = np.arctan2(dz, dx)   # angle w.r.t x-z plane
    psi   = np.arctan2(dy, dx)   # angle w.r.t x-y plane
    return start_node, end_node, L, e_m, theta, psi

# ------------------------
# Equivalent diameter normal to flow
# ------------------------
def equivalent_diameter(D, L, theta):
    """Compute projected equivalent diameter for diagonal member"""
    # Following lecture: D_eq = D * sin(theta) for diagonal members
    # For horizontal members: D_eq = D
    if np.isclose(theta, 0):
        return D
    else:
        return D * np.sin(abs(theta))

# ------------------------
# Stick-model aggregation
# ------------------------
def stick_model(nodes, members, H, T, h, rho=1025, CD=1.0, CM=2.0, t=0.0, zeta=0, dz=1.0, n_segments=10):
    """
    nodes: dict {node_id: (x,y,z)}
    members: list of dicts {'start':id, 'end':id, 'D':diameter}
    dz: vertical layer thickness for stick-model
    """
    # Determine vertical range of structure
    z_values = [coord[2] for coord in nodes.values()]
    z_min = min(z_values)
    z_max = max(z_values)
    layers = np.arange(z_min, z_max + dz, dz)
    
    # Initialize stick-model forces per layer
    Fx_layers = np.zeros_like(layers)
    Fz_layers = np.zeros_like(layers)
    
    # Loop over members
    for member in members:
        start, end, L, e_m, theta, psi = member_properties(nodes, member)
        D_eq = equivalent_diameter(member['D'], L, theta)
        
        # Discretize member into segments
        for i in range(n_segments):
            s = (i + 0.5) / n_segments
            point = start + s * (end - start)
            x, y, z = point
            
            if z > 0:
                continue  # above water
            
            u, w, u_dot, w_dot = wave_kinematics(x, z, t, H, T, h, zeta)
            vel = np.array([u, w])
            acc = np.array([u_dot, w_dot])
            
            # Normal components to member
            vel_n = vel - np.dot(vel, e_m[:2]) * e_m[:2]
            acc_n = acc - np.dot(acc, e_m[:2]) * e_m[:2]
            
            # Segment force
            f_seg = morison_segment_force(vel_n, acc_n, D_eq, rho, CD, CM)
            
            # Map to vertical layer index
            layer_idx = int((z - z_min) // dz)
            if 0 <= layer_idx < len(layers):
                Fx_layers[layer_idx] += f_seg[0] * (L / n_segments)
                Fz_layers[layer_idx] += f_seg[1] * (L / n_segments)
    
    # Return vertical stick-model forces
    stick_forces = []
    for i, z in enumerate(layers):
        stick_forces.append({'z': z, 'Fx': Fx_layers[i], 'Fz': Fz_layers[i]})
    
    # Total forces
    Fx_total = np.sum(Fx_layers)
    Fz_total = np.sum(Fz_layers)
    
    return Fx_total, Fz_total, stick_forces

# ------------------------
# Example usage
# ------------------------
if __name__ == "__main__":
    # Nodes in 3D space
    nodes = {
        1: (0, 0, -30),
        2: (10, 0, -5),
        3: (0, 0, 0),
        4: (10, 0, 0)
    }
    
    # Members connecting nodes
    members = [
        {'start': 1, 'end': 2, 'D': 1.5},  # diagonal
        {'start': 1, 'end': 3, 'D': 1.5},  # vertical
        {'start': 2, 'end': 4, 'D': 1.5}   # vertical
    ]
    
    H = 6.0     # wave height (m)
    T = 10.0    # wave period (s)
    h = 50.0    # water depth (m)
    zeta = H/2  # wave crest for Wheeler stretching
    
    Fx_total, Fz_total, stick_forces = stick_model(
        nodes, members, H, T, h, t=5.0, zeta=zeta, dz=1.0, n_segments=20
    )
    
    print("=== Total Forces on Structure ===")
    print(f"Fx_total = {Fx_total:.2f} N")
    print(f"Fz_total = {Fz_total:.2f} N\n")
    
    print("=== Stick-Model Forces per Layer ===")
    for f in stick_forces:
        if f['Fx'] !=0 or f['Fz'] !=0:
            print(f"z={f['z']:.1f} m: Fx={f['Fx']:.2f} N, Fz={f['Fz']:.2f} N")

=== Total Forces on Structure ===
Fx_total = -94218.98 N
Fz_total = 137542.46 N

=== Stick-Model Forces per Layer ===
z=-30.0 m: Fx=-915.19 N, Fz=3028.35 N
z=-29.0 m: Fx=-421.70 N, Fz=1408.63 N
z=-28.0 m: Fx=-613.77 N, Fz=1875.24 N
z=-27.0 m: Fx=-1170.34 N, Fz=3549.47 N
z=-26.0 m: Fx=-567.52 N, Fz=1604.27 N
z=-25.0 m: Fx=-1405.53 N, Fz=3926.19 N
z=-24.0 m: Fx=-1583.59 N, Fz=4215.34 N
z=-22.0 m: Fx=-1783.20 N, Fz=4516.24 N
z=-21.0 m: Fx=-2006.99 N, Fz=4829.44 N
z=-20.0 m: Fx=-1079.49 N, Fz=2118.23 N
z=-19.0 m: Fx=-2392.79 N, Fz=5260.07 N
z=-18.0 m: Fx=-1324.89 N, Fz=3271.91 N
z=-17.0 m: Fx=-1362.60 N, Fz=2326.98 N
z=-16.0 m: Fx=-3017.62 N, Fz=5950.78 N
z=-15.0 m: Fx=-3387.47 N, Fz=6316.03 N
z=-14.0 m: Fx=-1900.54 N, Fz=2628.80 N
z=-13.0 m: Fx=-1901.31 N, Fz=4066.02 N
z=-12.0 m: Fx=-4266.18 N, Fz=7087.26 N
z=-11.0 m: Fx=-2352.04 N, Fz=2811.13 N
z=-10.0 m: Fx=-5045.40 N, Fz=7574.83 N
z=-9.0 m: Fx=-5653.38 N, Fz=7985.64 N
z=-7.0 m: Fx=-6334.14 N, Fz=8407.57 N
z=-6.0 m: Fx=-7096.54 N, Fz=88